# 09.12 - Function / Tool Calling

**Phase:** 09 - Generative AI

**Status:** VERIFIED

---

## 1. What Are We Solving?

Function/tool calling lets LLMs invoke external functions by generating structured requests. The model decides which function to call and what arguments to pass, then your code executes the function and feeds the result back. This is the foundation of agents, RAG, and any app where the LLM must act on external data.

## 2. Why Does This Matter?

Tool calling bridges language understanding and real-world actions. Without it, the model can only 'think'; with it, the model can look up data, query APIs, and act on results.

## 3. Prerequisites

- Unit 09.11 (structured output)
- Unit 09.9 (LLM APIs)

## 4. Learning Objectives

- Define tools with name, description, and parameter schema
- Simulate the model deciding to call a tool
- Execute tools and return results to the model
- Handle tool errors and multi-turn tool use

## 5. Mental Model

Tool calling is like giving the LLM a phone book and letting it decide whom to call. The model reads the conversation, decides which tool is needed, formats a structured request, and your code executes the real function.

```text
user msg -> model -> tool_call(request) -> you execute -> tool result -> model final answer
```


## 6. Setup

No API key available; we simulate the model's tool-call decisions with a rule-based mock. The tool definitions and execution are real code - that is the production logic you control.


In [1]:
import matplotlib
matplotlib.use('Agg')
import json
print("Ready.")


Ready.


## 7. Define Real Tools

These are actual Python functions the assistant can call. In production they would hit weather/flight/db APIs.


In [2]:
def get_weather(location: str, units: str = "celsius") -> dict:
    """Get the current weather for a location."""
    # In production this calls a weather API.
    return {"location": location, "temp": 22, "condition": "sunny", "units": units}

def get_flight_price(from_city: str, to_city: str) -> dict:
    """Return the best available flight price between two cities."""
    return {"from": from_city, "to": to_city, "price": 249, "currency": "USD"}

# Test one tool directly
print(get_weather("Paris", "celsius"))


{'location': 'Paris', 'temp': 22, 'condition': 'sunny', 'units': 'celsius'}


## 8. Tool Schemas (for the model)

Describe each tool to the model with a name, description, and JSON parameter schema. The model uses these to decide and format calls.


In [3]:
TOOLS = [
    {
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "City name"},
                    "units": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["location"],
            },
        }
    },
    {
        "function": {
            "name": "get_flight_price",
            "description": "Return the best available flight price between two cities.",
            "parameters": {
                "type": "object",
                "properties": {
                    "from_city": {"type": "string"},
                    "to_city": {"type": "string"},
                },
                "required": ["from_city", "to_city"],
            },
        }
    },
]
print(json.dumps(TOOLS, indent=2))


[
  {
    "function": {
      "name": "get_weather",
      "description": "Get the current weather for a location.",
      "parameters": {
        "type": "object",
        "properties": {
          "location": {
            "type": "string",
            "description": "City name"
          },
          "units": {
            "type": "string",
            "enum": [
              "celsius",
              "fahrenheit"
            ]
          }
        },
        "required": [
          "location"
        ]
      }
    }
  },
  {
    "function": {
      "name": "get_flight_price",
      "description": "Return the best available flight price between two cities.",
      "parameters": {
        "type": "object",
        "properties": {
          "from_city": {
            "type": "string"
          },
          "to_city": {
            "type": "string"
          }
        },
        "required": [
          "from_city",
          "to_city"
        ]
      }
    }
  }
]


## 9. Mock Model's Tool Decision

Simulate the model returning a structured tool call. In production this comes from `client.chat.completions.create(..., tools=TOOLS, tool_choice="auto")`.


In [4]:
def mock_model_respond(user_msg):
    """Rule-based stand-in for the model's tool-call decision."""
    text = user_msg.lower()
    if "weather" in text:
        # extract a city if present
        cities = ["paris", "london", "tokyo", "new york"]
        city = next((c for c in cities if c in text), "London")
        return {"tool": "get_weather", "arguments": {"location": city.title()}}
    if "flight" in text or "price" in text:
        return {"tool": "get_flight_price", "arguments": {"from_city": "New York", "to_city": "London"}}
    return None   # no tool needed

decision = mock_model_respond("What's the weather in Paris?")
print("Model decided to call:", decision)


Model decided to call: {'tool': 'get_weather', 'arguments': {'location': 'Paris'}}


## 10. Execute the Tool Call

Your code validates arguments, executes the real function, and serializes the result for the model.


In [5]:
FN_MAP = {"get_weather": get_weather, "get_flight_price": get_flight_price}

def execute_tool_call(call):
    fn = FN_MAP[call["tool"]]
    result = fn(**call["arguments"])
    return json.dumps(result)

call = {"tool": "get_weather", "arguments": {"location": "Paris"}}
result_json = execute_tool_call(call)
print("Tool result:", result_json)


Tool result: {"location": "Paris", "temp": 22, "condition": "sunny", "units": "celsius"}


## 11. The Full Tool-Calling Loop

Combine: user message -> model decides -> execute -> feed result back -> final answer. This is the core agent loop.


In [6]:
def run_agent(user_msg):
    # Step 1: model decides (mock)
    decision = mock_model_respond(user_msg)
    if decision is None:
        return f"No tool needed. Direct answer to: {user_msg}"

    # Step 2: execute
    tool_result = execute_tool_call(decision)

    # Step 3: (mock) model reads the tool result and produces a final answer
    final = f"Using {decision['tool']} result {tool_result}, here is the answer."
    return final

print(run_agent("What's the weather in Tokyo?"))
print()
print(run_agent("How much is a flight from New York to London?"))
print()
print(run_agent("Tell me a joke."))


Using get_weather result {"location": "Tokyo", "temp": 22, "condition": "sunny", "units": "celsius"}, here is the answer.

Using get_flight_price result {"from": "New York", "to": "London", "price": 249, "currency": "USD"}, here is the answer.

No tool needed. Direct answer to: Tell me a joke.


## 12. Error Handling for Tool Calls

Tools can fail (API down, bad args). Wrap execution and return a helpful error to the model or user.


In [7]:
def safe_execute(call):
    try:
        return execute_tool_call(call)
    except KeyError as e:
        return json.dumps({"error": f"Unknown tool: {e}"})
    except TypeError as e:
        return json.dumps({"error": f"Bad arguments: {e}"})

print("Bad tool:", safe_execute({"tool": "nope", "arguments": {}}))
print("Bad args:", safe_execute({"tool": "get_weather", "arguments": {}}))
print("\nAlways validate + handle errors so a failed tool call doesn't crash the app.")


Bad tool: {"error": "Unknown tool: 'nope'"}
Bad args: {"error": "Bad arguments: get_weather() missing 1 required positional argument: 'location'"}

Always validate + handle errors so a failed tool call doesn't crash the app.


## 13. Debugging Tool Calls

| Symptom | Cause | Fix |
|---|---|---|
| Model does not call tool | description unclear | improve description/examples |
| Wrong arguments | parameter schema unclear | add descriptions/enums |
| Ignores tool result | result not in conversation | format tool response correctly |
| Calls wrong tool | similar tools | make names/descriptions distinct |

## 14. Real-World Considerations

- Validate tool arguments before executing (model may hallucinate).
- Log every tool call and result for debugging.
- Use clear, specific tool names/descriptions.
- `tool_choice`: "auto" lets model decide; "required"/specific name forces a call.

## 15. Common Mistakes

- Not handling tool call errors.
- Trusting model output blindly.
- Not validating arguments.
- Missing tool results in the conversation (model cannot reason about them).

## 16. When NOT to Use Tool Calling

- Pure text generation (no external action needed).
- Safety-critical actions -> use strict manual tool selection instead.
- Deterministic tool use -> skip the model and call directly.

## 17. Challenge

Chain two tool calls: ask for the weather in both Paris and Tokyo in one request and return both results.


In [8]:
def run_multi_tool_agent(user_msg):
    # Simulate model issuing two tool calls
    calls = [
        {"tool": "get_weather", "arguments": {"location": "Paris"}},
        {"tool": "get_weather", "arguments": {"location": "Tokyo"}},
    ]
    results = [safe_execute(c) for c in calls]
    return f"Got {len(calls)} results: {results}"

print(run_multi_tool_agent("Weather in Paris and Tokyo?"))
print("\nParallel tool calls improve latency in production (batch round-trips).")


Got 2 results: ['{"location": "Paris", "temp": 22, "condition": "sunny", "units": "celsius"}', '{"location": "Tokyo", "temp": 22, "condition": "sunny", "units": "celsius"}']

Parallel tool calls improve latency in production (batch round-trips).


## 18. Closed-Book Recall

1. How does the model decide which tool to call?
2. What happens when a tool call fails?
3. How do you return tool results to the model?
4. What is the difference between tool_choice 'auto' and 'required'?

## 19. Teach-Back Questions

Explain to another person:

- The agent loop: decide -> execute -> return -> answer.
- Why you must validate tool arguments.

## 20. Summary

You defined real tools with schemas, simulated the model's tool-call decision, executed calls, handled errors, and chained multiple calls - the core agent pattern.

## 21. Further Experiment

- Connect `get_weather` to a real weather API when available.
- Implement tool_choice semantics (auto vs required) in the mock.

## 22. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: none required at runtime (mock model), json
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
